In [ ]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
import torch
from torch import nn, optim
import argparse
import sys
sys.path.append('./')
sys.path.append('../')
import os
import yaml
from easydict import EasyDict
from collections import OrderedDict
import random
import numpy as np
import pickle as pkl
import h5py

from torch.utils.data import IterableDataset

from Data.Transition1x import generate_dataloader_dynamics
from Model.model import DistFlowMatchingNetwork, ODEWrapper2, InterpNetwork
from Model.backbone import generate_backbone
from Model.head import generate_head
from Model.model import MDNet
from Utils import pairwise_dist_to_coord, get_logger, get_new_log_dir, seed_all, Kabsch_alignment, rmsd_loss, generate_fully_connected, create_angular_index, calculate_angle, d_mae_loss, calculate_efh, AU2EV, MLCalculator, PyscfCalculator, neb, Sella_Opt, count_negative_eig
import gc

from copy import deepcopy

from torch_scatter import scatter_mean, scatter_add
from torch.optim import LBFGS

from torch_geometric.data import Data, DataLoader

import pickle

from sella import Sella

In [ ]:
from ase import Atoms
from ase.mep.neb import NEB, NEBTools, NEBOptimizer
from ase.optimize import MDMin, BFGS
from ase.calculators.calculator import Calculator, all_changes

In [ ]:
parser = argparse.ArgumentParser(description='Training Transition1x dynamics')
parser.add_argument('--config_file_flow', required=True)
parser.add_argument('--config_file_potential', required=True)
parser.add_argument('--log_prefix', default='logs')
parser.add_argument('--notes', default=' ')
parser.add_argument('--device', default='cuda')
parser.add_argument('--resume_status', default=' ')
parser.add_argument('--potential', default=' ')
parser.add_argument('--flow', default=' ')
args = parser.parse_args(['--config_file_flow', "../Configs/Dynamics.yml",
                          '--device', 'cuda',
                          '--config_file_potential', '../Configs/Potential.yml',
                          '--flow', '', # file to trained TS-DFM checkpoint
                          '--potential', '']) # file to trained MLIP checkpoint

In [ ]:
def calculate_loss_coord(adj_matrix, coord, edge_index):
    src = edge_index[0]
    dst = edge_index[1]
    diff = coord[src] - coord[dst]
    dists = torch.norm(diff, p=2, dim=-1)
    loss = torch.sum(torch.square(dists - adj_matrix) * torch.pow(1 / (adj_matrix + 1e-6), 2))
    return loss

In [ ]:
dtype = torch.float32

config_path=args.config_file_potential
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
config = EasyDict(config)
config.notes = args.notes

device = args.device

In [ ]:
seed_all(config.train.seed)
torch.backends.cudnn.benchmark = True

In [ ]:
backbone = generate_backbone(config.model.backbone)
head = generate_head(config.model.head)

In [ ]:
REFERENCE_ENERGIES = {
    1: -13.62222753701504,
    6: -1029.4130839658328,
    7: -1484.8710358098756,
    8: -2041.8396277138045,
    9: -2712.8213146878606,
}

In [ ]:
potential_model = MDNet(backbone, head, REFERENCE_ENERGIES)
best_state = torch.load(args.potential, map_location=device)
potential_model.load_state_dict(best_state['model'])
potential_model.eval()

In [ ]:
config_path=args.config_file_flow
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
config = EasyDict(config)

In [ ]:
config.data.batch_size = 1

In [ ]:
dynamic_model = DistFlowMatchingNetwork(**config.dynamic_model.parameters)
dynamic_model.load_state_dict(torch.load(args.flow)['model'])

In [ ]:
def get_molecular_reference_energy(atomic_numbers):
    molecular_reference_energy = 0
    for atomic_number in atomic_numbers:
        molecular_reference_energy += REFERENCE_ENERGIES[atomic_number]

    return molecular_reference_energy

def generator(formula, rxn, grp):
    """ Iterates through a h5 group """

    energies = grp["wB97x_6-31G(d).energy"]
    forces = grp["wB97x_6-31G(d).forces"]
    atomic_numbers = list(grp["atomic_numbers"])
    positions = grp["positions"]
    molecular_reference_energy = get_molecular_reference_energy(atomic_numbers)

    for energy, force, positions in zip(energies, forces, positions):
        d = {
            "rxn": rxn,
            "wB97x_6-31G(d).energy": energy.__float__(),
            "wB97x_6-31G(d).atomization_energy": energy
            - molecular_reference_energy.__float__(),
            "wB97x_6-31G(d).forces": force.tolist(),
            "positions": positions,
            "formula": formula,
            "atomic_numbers": atomic_numbers,
        }

        yield d

def get_dynamics_data(formula, rxn, data):
    reactant = next(generator(formula, rxn, data[formula][rxn]["reactant"]))
    product = next(generator(formula, rxn, data[formula][rxn]["product"]))
    transition_state = next(generator(formula, rxn, data[formula][rxn]["transition_state"]))
    x = torch.tensor(reactant['atomic_numbers'], dtype=torch.long)

    reactant_pos = torch.tensor(reactant['positions'], dtype=torch.float32)
    product_pos = torch.tensor(product['positions'], dtype=torch.float32)
    transition_state_pos = torch.tensor(transition_state['positions'], dtype=torch.float32)
    product_pos = Kabsch_alignment(product_pos, reactant_pos, torch.zeros_like(x))
    transition_state_pos = Kabsch_alignment(transition_state_pos, reactant_pos, torch.zeros_like(x))

    energies = list()
    energies.append(torch.tensor(reactant['wB97x_6-31G(d).energy'], dtype=torch.float32))
    energies.append(torch.tensor(product['wB97x_6-31G(d).energy'], dtype=torch.float32))
    energies.append(torch.tensor(transition_state['wB97x_6-31G(d).energy'], dtype=torch.float32))
    
    return Data(x=x, reactant_pos=reactant_pos, product_pos=product_pos, transition_state_pos=transition_state_pos, energies=torch.stack(energies))

class Dataset_dynamics(IterableDataset):
    def __init__(self, hdf5_file, datasplit):
        super(Dataset_dynamics, self).__init__()
        self.hdf5_file = hdf5_file
        self.datasplit = datasplit
        assert datasplit in [
            "train",
            "valid",
            "test",
        ]
        with open('../Data/reactions_'+self.datasplit+'.pickle', 'rb') as f:
            self.datalist = pickle.load(f)

    def __iter__(self):
        with h5py.File(self.hdf5_file, "r") as f:
            data = f['data']
            i = 0
            if self.datasplit == 'train':
                random.shuffle(self.datalist)
            for formula, rxn in self.datalist:
                yield get_dynamics_data(formula, rxn, data)
                    
    def __len__(self):
        pass
    
def generate_dataloader_dynamics(hdf5_file, batch_size):
    dataloaders = {}
    dataloaders['train'] = DataLoader(dataset=Dataset_dynamics(hdf5_file, 'train'), batch_size = batch_size)
    dataloaders['val'] = DataLoader(dataset=Dataset_dynamics(hdf5_file, 'valid'), batch_size = batch_size)
    dataloaders['test'] = DataLoader(dataset=Dataset_dynamics(hdf5_file, 'test'), batch_size = batch_size)
    return dataloaders

In [ ]:
dataloaders = generate_dataloader_dynamics('transition1x.h5', config.data.batch_size)

In [ ]:
dynamic_model = dynamic_model.to(device)
potential_model = potential_model.to(device)
ode = ODEWrapper2(dynamic_model)

In [ ]:
dynamic_model.eval()
potential_model.eval()

In [ ]:
calculator_ml = MLCalculator(potential_model)
calculator_dft = PyscfCalculator(device='cuda')

In [ ]:
neb_res = []
true_trans_pos = []
pred_trans_pos_fm = []
pred_trans_pos_neb = []
true_pot_energy = []
success_neb = []
dft_res_neb = []
dft_res_fm = []

In [ ]:
def pathpred(data):
    timestep = 0.1

    batch = data.batch.to(device)
    x = data.x.to(device)
    reactant_pos = data.reactant_pos.to(device)
    product_pos = data.product_pos.to(device)

    transition_state_pos = data.transition_state_pos.to(device)

    src, dst = generate_fully_connected(batch)


    edge_index = torch.concat([src.unsqueeze(0), dst.unsqueeze(0)], dim=0)

    dist_reactant = torch.norm(reactant_pos[src] - reactant_pos[dst], p=2, dim=-1)
    dist_product = torch.norm(product_pos[src] - product_pos[dst], p=2, dim=-1)

    interp_t = torch.ones_like(batch, dtype=torch.float32) * 0.5

    dist_lin_interp = (1 - interp_t[batch[src]]) * torch.norm(reactant_pos[src] - reactant_pos[dst], p=2, dim=-1) + interp_t[batch[src]] * torch.norm(product_pos[src] - product_pos[dst], p=2, dim=-1)

    dist_transition_state = torch.norm(transition_state_pos[src] - transition_state_pos[dst], p=2, dim=-1)

    dist_transition_state_pred = ode(x, edge_index, dist_reactant, dist_product, dist_lin_interp, batch)

    curr_dist_matrix = dist_transition_state_pred.detach()
    
    pred_pos, opt_loss = pairwise_dist_to_coord(x, reactant_pos, product_pos, curr_dist_matrix)
    pred_pos = pred_pos.detach()

    pred_pos = Kabsch_alignment(pred_pos, transition_state_pos, torch.zeros(pred_pos.shape[0], dtype=torch.int64, device=device))

    atoms = Atoms(numbers=x.cpu().numpy(), positions=pred_pos.detach().cpu().numpy())
    pred_trans_pos_fm.append(atoms.get_positions())

    # energy_dft_fm = 0.0
    dft_res_fm.append(calculate_efh(atoms, f=True, hess=True, return_metrics=True))
    energy_dft_fm = dft_res_fm[-1][0].e_tot * AU2EV

    pred_pos = torch.tensor(atoms.get_positions(), dtype=torch.float32).to(device)
    loss_rmsd_fm = rmsd_loss(pred_pos, transition_state_pos, torch.zeros(pred_pos.shape[0], dtype=torch.int64, device=device))
    loss_dmae_fm = d_mae_loss(pred_pos, transition_state_pos, torch.zeros(pred_pos.shape[0], dtype=torch.int64, device=device))

    energy_pred_fm, _ = potential_model(x, pred_pos, None, None, batch)

    atom_configs, is_success_neb = neb(calculator_ml, x, reactant_pos, product_pos, pred_pos)
    success_neb.append(is_success_neb)

    max_energy_ind = 0
    max_energy = -10000000.0

    for j in range(len(atom_configs)):
        atoms = atom_configs[j]
        x = torch.tensor(atoms.get_atomic_numbers()).to(device)
        pos = torch.tensor(atoms.get_positions(), dtype=torch.float32).to(device)
        batch = torch.zeros_like(x).to(device)
        energy, force = potential_model.get_energy_and_force(x, pos, None, None, batch)
        energy = energy.cpu()
        if energy > max_energy:
            max_energy = energy
            max_energy_ind = j

    atoms = atom_configs[max_energy_ind]

    # energy_dft = 0.0
    dft_res_neb.append(calculate_efh(atoms, f=True, hess=True, return_metrics=True))
    energy_dft = dft_res_neb[-1][0].e_tot * AU2EV
    pred_trans_pos_neb.append(atoms.get_positions())

    pred_pos = torch.tensor(atoms.get_positions(), dtype=torch.float32).to(device)
    loss_rmsd = rmsd_loss(pred_pos, transition_state_pos, torch.zeros(pred_pos.shape[0], dtype=torch.int64, device=device))
    loss_dmae = d_mae_loss(pred_pos, transition_state_pos, torch.zeros(pred_pos.shape[0], dtype=torch.int64, device=device))
    energy_pred = max_energy
    return (loss_rmsd_fm.cpu(), loss_dmae_fm.cpu(), energy_pred_fm.detach().cpu(), energy_dft_fm), (loss_rmsd.cpu(), loss_dmae.cpu(), energy_pred.detach().cpu(), energy_dft) 

In [ ]:
energy_diffs = []
energy_diffs_dft = []
rmsds = []
dmaes = []

energy_diffs_fm = []
energy_diffs_dft_fm = []
rmsds_fm = []
dmaes_fm = []

for data in dataloaders['test']:
    energy = data.energies.numpy()
    transition_state_pos = data.transition_state_pos.to(device)
    temp1, temp2 = pathpred(data)

    loss_rmsd_fm, loss_dmae_fm, energy_pred_fm, energy_dft_fm = temp1
    loss_rmsd, loss_dmae, energy_pred, energy_dft = temp2

    energy_diffs_fm.append(np.abs(energy_pred_fm - energy[-1]))
    energy_diffs_dft_fm.append(np.abs(energy_dft_fm - energy[-1]))
    rmsds_fm.append(loss_rmsd_fm)
    dmaes_fm.append(loss_dmae_fm)

    energy_diffs.append(np.abs(energy_pred - energy[-1]))
    energy_diffs_dft.append(np.abs(energy_dft - energy[-1]))
    rmsds.append(loss_rmsd)
    dmaes.append(loss_dmae)


In [ ]:
rmsds = torch.concat(rmsds)
dmaes = torch.concat(dmaes)
# energy_diffs = torch.vstack(energy_diffs)
energy_diffs_dft = torch.tensor(energy_diffs_dft)

rmsds_fm = torch.concat(rmsds_fm)
dmaes_fm = torch.concat(dmaes_fm)
# energy_diffs_fm = torch.vstack(energy_diffs_fm)
energy_diffs_dft_fm = torch.tensor(energy_diffs_dft_fm)

In [ ]:
import pickle
with open('res_fm&neb.pickle', 'wb') as f:
    pickle.dump({
        'neb': {
            'rmsd': rmsds,
            'dmae': dmaes,
            'barriers_model': energy_diffs,
            'barriers_dft': energy_diffs_dft,
            'success': success_neb,
            'pred_trans_pos': pred_trans_pos_neb,
            'dft_res': dft_res_neb
        },
        'fm': {
            'rmsd': rmsds_fm,
            'dmae': dmaes_fm,
            'barriers_model': energy_diffs_fm,
            'barriers_dft': energy_diffs_dft_fm,
            'pred_trans_pos': pred_trans_pos_fm,
            'dft_res': dft_res_fm
        },
    }, f)